# House Price Prediction

End-to-end workflow for EDA, preprocessing, feature engineering, modeling, and evaluation.

## Setup
Imports and plotting style.

In [ ]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set plotting style and random seed
sns.set_theme(style="whitegrid")
np.random.seed(42)

## Data Loading
Load Kaggle CSV if present; otherwise generate synthetic sample.

In [ ]:
# Load CSV if available; otherwise build a synthetic dataset
possible_paths = [
    "house_price_prediction.csv",
    "house_price.csv",
    os.path.join("data", "house_price_prediction.csv"),
    os.path.join("data", "house_price.csv"),
]

df = None
for path in possible_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"Loaded dataset from: {path}")
        break

if df is None:
    rng = np.random.default_rng(42)
    n_rows = 500

    area = rng.normal(1500, 500, n_rows).clip(300, 4000)
    bedrooms = rng.integers(1, 6, n_rows)
    bathrooms = rng.integers(1, 4, n_rows)
    floors = rng.integers(1, 3, n_rows)
    year_built = rng.integers(1970, 2024, n_rows)
    furnishing = rng.choice(
        ["furnished", "semi-furnished", "unfurnished"],
        n_rows,
        p=[0.3, 0.4, 0.3],
    )

    price = (
        area * 3000
        + bedrooms * 10000
        + bathrooms * 15000
        + floors * 5000
        + (year_built - 1970) * 200
        + rng.normal(0, 50000, n_rows)
    )

    df = pd.DataFrame({
        "Price": price,
        "Area": area,
        "Bedrooms": bedrooms,
        "Bathrooms": bathrooms,
        "Floors": floors,
        "YearBuilt": year_built,
        "Furnishing": furnishing,
    })

    # Inject missing values to simulate real-world data
    for col in ["Area", "Bedrooms", "Bathrooms", "YearBuilt", "Furnishing"]:
        missing_idx = rng.choice(df.index, size=int(0.05 * n_rows), replace=False)
        df.loc[missing_idx, col] = np.nan

    print("CSV not found. Using synthetic dataset.")

df.head()

## 1. Exploratory Data Analysis (EDA)
Inspect structure, missing data, and summary statistics.

In [ ]:
# Basic dataset information
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)

# Missing value analysis
missing_values = df.isna().sum().sort_values(ascending=False)
print("\nMissing values:\n", missing_values)

# Statistical summary (mean, median, std)
numeric_cols = df.select_dtypes(include="number").columns
stats = df[numeric_cols].agg(["mean", "median", "std"]).T
print("\nSummary statistics:\n", stats)

In [ ]:
# Histograms for numeric features
df[numeric_cols].hist(figsize=(12, 8), bins=20)
plt.tight_layout()
plt.show()

# Boxplots for numeric features
plt.figure(figsize=(12, 6))
sns.boxplot(data=df[numeric_cols], orient="h")
plt.tight_layout()
plt.show()

# Scatter plots: Price vs key features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.scatterplot(ax=axes[0], data=df, x="Area", y="Price")
sns.scatterplot(ax=axes[1], data=df, x="Bedrooms", y="Price")
sns.scatterplot(ax=axes[2], data=df, x="Bathrooms", y="Price")
for ax in axes:
    ax.set_xlabel(ax.get_xlabel())
    ax.set_ylabel("Price")
plt.tight_layout()
plt.show()

# Correlation heatmap
corr = df[numeric_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.tight_layout()
plt.show()

In [ ]:
# Outlier detection using IQR
outlier_counts = {}
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_counts[col] = ((df[col] < lower) | (df[col] > upper)).sum()

outliers = pd.Series(outlier_counts).sort_values(ascending=False)
print("Outlier counts (IQR method):\n", outliers)

## 2. Preprocessing
Handle missing values, encode categorical columns, and prepare features/target.

In [ ]:
# Copy data for cleaning
df_clean = df.copy()

# Fill missing values: median for numeric, mode for categorical
num_cols = df_clean.select_dtypes(include="number").columns
cat_cols = df_clean.select_dtypes(exclude="number").columns
for col in num_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
for col in cat_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode(dropna=True)[0])

# Encode categorical variables
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

# Separate features and target
target = "Price"
X = df_encoded.drop(columns=[target])
y = df_encoded[target]

## 3. Feature Engineering
Select top correlated features and set up polynomial features.

In [ ]:
# Identify top correlated features with the target
correlations = df_encoded.corr(numeric_only=True)[target].sort_values(ascending=False)
top_features = correlations.drop(target).head(5)
print("Top correlated features:\n", top_features)

# Configure polynomial features (degree=2) for polynomial regression
poly = PolynomialFeatures(degree=2, include_bias=False)

## 4. Train/Test Split and Scaling
Split data and apply StandardScaler without leakage.

In [ ]:
# Split data into train and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create polynomial features from scaled data
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

## 5. Model Training
Train Linear, Polynomial, and Ridge Regression models.

In [ ]:
# Train Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)

# Train Polynomial Regression (degree=2)
poly_reg = LinearRegression()
poly_reg.fit(X_train_poly, y_train)

# Train Ridge Regression
ridge_reg = Ridge(alpha=1.0)
ridge_reg.fit(X_train_scaled, y_train)

## 6. Evaluation and Comparison
Compute RMSE, MAE, and $R^2$ for all models.

In [ ]:
# Evaluate models and build comparison table
def evaluate_model(name, model, X_eval, y_eval):
    preds = model.predict(X_eval)
    rmse = mean_squared_error(y_eval, preds, squared=False)
    mae = mean_absolute_error(y_eval, preds)
    r2 = r2_score(y_eval, preds)
    return {"Model": name, "RMSE": rmse, "MAE": mae, "R2": r2}, preds

results = []
predictions = {}

row, preds = evaluate_model("Linear Regression", lin_reg, X_test_scaled, y_test)
results.append(row)
predictions["Linear Regression"] = preds

row, preds = evaluate_model("Polynomial Regression", poly_reg, X_test_poly, y_test)
results.append(row)
predictions["Polynomial Regression"] = preds

row, preds = evaluate_model("Ridge Regression", ridge_reg, X_test_scaled, y_test)
results.append(row)
predictions["Ridge Regression"] = preds

comparison_df = pd.DataFrame(results).sort_values("RMSE")
print(comparison_df)

## 7. Visualizations
Predicted vs actual, residuals, RMSE comparison, and coefficients.

In [ ]:
# Predicted vs actual scatter plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models = ["Linear Regression", "Polynomial Regression", "Ridge Regression"]
for ax, name in zip(axes, models):
    ax.scatter(y_test, predictions[name], alpha=0.7)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
    ax.set_title(name)
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
plt.tight_layout()
plt.show()

# Residuals plot for ridge regression
residuals = y_test - predictions["Ridge Regression"]
plt.figure(figsize=(6, 4))
plt.scatter(predictions["Ridge Regression"], residuals, alpha=0.7)
plt.axhline(0, color="r", linestyle="--")
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Ridge Regression Residuals")
plt.tight_layout()
plt.show()

# Bar chart comparing RMSE
plt.figure(figsize=(6, 4))
sns.barplot(x="Model", y="RMSE", data=comparison_df)
plt.xticks(rotation=20, ha="right")
plt.title("RMSE Comparison")
plt.tight_layout()
plt.show()

# Feature coefficient plot (top 10 by absolute value from Ridge)
coef_series = pd.Series(ridge_reg.coef_, index=X.columns)
top_coef = coef_series.reindex(coef_series.abs().sort_values(ascending=False).head(10).index)
plt.figure(figsize=(8, 4))
top_coef.sort_values().plot(kind="barh")
plt.title("Top Ridge Coefficients")
plt.tight_layout()
plt.show()